<a href="https://colab.research.google.com/github/shashidhar078/FlipGrid/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

!pip install catboost
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00


In [3]:
train = pd.read_csv("processed_train_v1.csv")
test = pd.read_csv("processed_test_v1.csv")

y = train["demand"]
X = train.drop("demand", axis=1)

In [6]:
cat_model = CatBoostRegressor(
    depth=9,
    iterations=1299,
    l2_leaf_reg=2.74,
    learning_rate=0.038,
    random_strength=0.16,
    loss_function="RMSE",
    random_state=42,
    verbose=0
)

# Dynamically identify all object-type columns to treat as categorical features
categorical_features = X.select_dtypes(include='object').columns.tolist()
cat_model.fit(X, y, cat_features=categorical_features)
cat_pred = cat_model.predict(test)

In [8]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

# Identify categorical columns for one-hot encoding
categorical_cols = X.select_dtypes(include='object').columns

# Apply one-hot encoding to training data (X)
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Apply one-hot encoding to test data (test)
test_encoded = pd.get_dummies(test, columns=categorical_cols, drop_first=True)

# Align columns to ensure both datasets have the same columns after one-hot encoding
# This handles cases where test data might have categories not present in train, or vice-versa
X_encoded, test_encoded = X_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

rf_model.fit(X_encoded, y)
rf_pred = rf_model.predict(test_encoded)

In [9]:
final_pred = (
    0.65 * cat_pred +
    0.35 * rf_pred
)

final_pred = np.clip(final_pred, 0, 1)

In [11]:
submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": final_pred
})

submission.to_csv("submission.csv", index=False)

print("MAIN FILE EXECUTED SUCCESSFULLY")


MAIN FILE EXECUTED SUCCESSFULLY
